# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/31-PythonCNNGoruntuSiniflandirma.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 31 - Python ile CNN ve Görüntü Sınıflandırma

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bir önceki derste Dense katmanlardan oluşan ilk yapay sinir ağlarımızı kurduk.

Bu derste görüntüler için özel olarak tasarlanmış **Convolutional Neural Network (CNN)** yapılarına geçiyoruz.

Ana hedefimiz:

**8×8 el yazısı rakam görüntülerini CNN ile sınıflandıran çalışan bir görüntü yapay zekası geliştirmek.**

Bu derste:

- convolution,
- kernel / filter,
- stride,
- padding,
- feature map,
- Conv2D,
- MaxPooling2D,
- Flatten,
- Dropout,
- CNN mimarisi,
- görüntü tensörü,
- veri normalizasyonu,
- train / validation / test,
- learning curve,
- confusion matrix,
- yanlış tahmin analizi,
- feature map görselleştirme,
- data augmentation,
- model kaydetme ve yükleme

konularını uygulamalı olarak öğreneceğiz.


# 1. Dense Ağ ile Görüntü İşlemenin Sınırı

Önceki derste 8×8 görüntüyü:

```text
8 × 8
↓
64 piksel
↓
Flatten
↓
Dense
```

şeklinde modele verdik.

Bu yaklaşım çalışabilir ancak görüntünün iki boyutlu uzamsal yapısını doğrudan özel bir işlemle kullanmaz.

Örneğin:

```text
yan yana bulunan pikseller
kenarlar
köşeler
küçük şekiller
```

görüntü için önemlidir.

CNN bu yerel örüntüleri öğrenmek için convolution katmanlarını kullanır.


# 2. CNN Nedir?

CNN, özellikle görüntü gibi düzenli ızgara yapısındaki verilerde kullanılan sinir ağı mimarilerinden biridir.

Basit CNN:

```text
Görüntü
↓
Conv2D
↓
ReLU
↓
MaxPooling
↓
Conv2D
↓
ReLU
↓
MaxPooling
↓
Flatten
↓
Dense
↓
Softmax
```

şeklinde olabilir.


# 3. Convolution Mantığı

Convolution işleminde küçük bir matris görüntünün üzerinde dolaştırılır.

Bu küçük matrise:

- kernel,
- filter

denebilir.

Örnek:

```text
Görüntü
5 × 5

Kernel
3 × 3
```

Kernel görüntünün küçük bölgeleriyle matematiksel işlem yaparak yeni bir **feature map** oluşturur.


# 4. Basit Görüntü Matrisi Oluşturalım

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

goruntu = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
], dtype=float)

print(goruntu)


# 5. Basit Kernel

Kenar benzeri değişimleri yakalamaya çalışan örnek bir kernel oluşturalım.


In [ ]:
kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
], dtype=float)

print(kernel)


# 6. Tek Bir Bölge Üzerinde İşlem

Görüntünün sol üst 3×3 bölgesini alalım.


In [ ]:
bolge = goruntu[
    0:3,
    0:3
]

print(bolge)


Element bazında çarpıp sonuçları toplayalım.


In [ ]:
sonuc = np.sum(
    bolge
    *
    kernel
)

print(
    "Convolution sonucu:",
    sonuc
)


# 7. Kernel'i Görüntü Üzerinde Kaydırmak

Şimdi aynı işlemi görüntünün bütün uygun bölgelerinde yapalım.


In [ ]:
def basit_convolution(
    image,
    kernel
):
    h, w = image.shape
    kh, kw = kernel.shape

    cikti_h = (
        h
        -
        kh
        +
        1
    )

    cikti_w = (
        w
        -
        kw
        +
        1
    )

    cikti = np.zeros(
        (
            cikti_h,
            cikti_w
        ),
        dtype=float
    )

    for y in range(
        cikti_h
    ):
        for x in range(
            cikti_w
        ):
            bolge = image[
                y:y + kh,
                x:x + kw
            ]

            cikti[
                y,
                x
            ] = np.sum(
                bolge
                *
                kernel
            )

    return cikti


In [ ]:
feature_map = basit_convolution(
    goruntu,
    kernel
)

print(
    feature_map
)


# 8. Feature Map Nedir?

Kernel görüntünün farklı bölgelerine uygulandığında yeni bir matris oluşur.

Bu matrise **feature map / özellik haritası** denir.

CNN içindeki convolution katmanları:

```text
kenar
çizgi
doku
şekil parçaları
```

gibi faydalı örüntüleri öğrenebilen filtreler oluşturabilir.


# 9. Feature Map'i Görselleştirmek

In [ ]:
plt.imshow(
    feature_map,
    cmap="gray"
)

plt.title("Örnek Feature Map")
plt.axis("off")
plt.show()


# 10. Kernel Ağırlıkları Sabit midir?

Az önce kernel değerlerini biz belirledik.

Gerçek CNN eğitiminde Conv2D katmanındaki kernel ağırlıkları:

```text
başlangıç değerleri
↓
loss
↓
backpropagation
↓
optimizer
↓
öğrenilen filtreler
```

süreciyle model tarafından öğrenilir.

CNN'in önemli gücü budur.


# 11. Birden Fazla Filter

Bir Conv2D katmanında tek kernel kullanmak zorunda değiliz.

Örneğin:

```python
Conv2D(
    filters=32,
    kernel_size=(3, 3)
)
```

32 farklı filter öğrenir.

Sonuçta 32 farklı feature map oluşabilir.


# 12. Stride Nedir?

Kernel'in görüntü üzerinde her adımda kaç piksel ilerleyeceğini belirler.

```text
stride = 1
```

birer piksel,

```text
stride = 2
```

ikişer piksel ilerleme anlamına gelir.

Stride büyüdükçe çıktı boyutu küçülebilir.


# 13. Padding Nedir?

Convolution uygulandığında görüntünün kenarlarında boyut küçülebilir.

`padding="same"` yaklaşımı çıktı uzamsal boyutunu uygun stride durumunda giriş boyutuna yakın veya aynı tutmak için kenarlara padding ekler.

`padding="valid"` ise padding eklemeden yalnızca geçerli bölgelerde convolution uygular.


# 14. 5×5 Girdiye 3×3 Kernel

Padding yok, stride 1 ise:

```text
Girdi  = 5×5
Kernel = 3×3
Çıktı  = 3×3
```

olur.

Az önce elle yaptığımız örnek bunun bir örneğidir.


# 15. Max Pooling Nedir?

Pooling feature map'in uzamsal boyutunu küçültmek için kullanılır.

Max Pooling belirli pencere içindeki en büyük değeri alır.

Örneğin:

```text
1  4
2  3

↓ Max

4
```


# 16. Basit Max Pooling Örneği

In [ ]:
pool_girdi = np.array([
    [1, 3, 2, 1],
    [4, 6, 5, 2],
    [1, 2, 8, 7],
    [0, 3, 4, 9]
])

print(
    pool_girdi
)


2×2 pencere ve stride 2 ile sonucu elle oluşturalım.


In [ ]:
pool_cikti = np.array([
    [
        np.max(
            pool_girdi[
                0:2,
                0:2
            ]
        ),
        np.max(
            pool_girdi[
                0:2,
                2:4
            ]
        )
    ],
    [
        np.max(
            pool_girdi[
                2:4,
                0:2
            ]
        ),
        np.max(
            pool_girdi[
                2:4,
                2:4
            ]
        )
    ]
])

print(
    pool_cikti
)


# 17. Pooling Neden Kullanılır?

Pooling:

- feature map boyutunu küçültebilir,
- hesaplama maliyetini azaltabilir,
- küçük konum değişikliklerine karşı daha dayanıklı temsil üretmeye yardımcı olabilir.

Ancak aşırı pooling önemli ayrıntıları kaybettirebilir.


# 18. CNN Girdi Şekli

Keras'ta varsayılan `channels_last` düzeninde görüntü batch'i çoğunlukla:

```text
(batch, height, width, channels)
```

şeklindedir.

Örnek:

```text
32 adet RGB görüntü
32 × 64 × 64 × 3
```

Gri görüntü:

```text
32 × 64 × 64 × 1
```


# 19. TensorFlow ve Keras

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(
    "TensorFlow:",
    tf.__version__
)

print(
    "Keras:",
    keras.__version__
)


# 20. Seed Belirlemek

In [ ]:
keras.utils.set_random_seed(
    42
)


# 21. Keras Conv2D ile Tek Örnek

5×5 görüntümüzü 4 boyutlu tensöre dönüştürelim.


In [ ]:
cnn_ornek = goruntu.reshape(
    1,
    5,
    5,
    1
).astype(
    np.float32
)

print(
    cnn_ornek.shape
)


# 22. Conv2D Katmanı Oluşturmak

In [ ]:
conv_ornek = layers.Conv2D(
    filters=2,
    kernel_size=3,
    activation="relu",
    padding="valid"
)

conv_cikti = conv_ornek(
    cnn_ornek
)

print(
    conv_cikti.shape
)


Girdi:

```text
1 × 5 × 5 × 1
```

Çıktı:

```text
1 × 3 × 3 × 2
```

olur.

Son boyuttaki `2`, iki farklı filter'ın ürettiği feature map sayısıdır.


# 23. Conv2D Ağırlıklarının Şekli

In [ ]:
agirliklar = (
    conv_ornek.get_weights()
)

print(
    "Kernel shape:",
    agirliklar[0].shape
)

print(
    "Bias shape:",
    agirliklar[1].shape
)


Kernel ağırlık şekli genel olarak:

```text
(kernel_height,
 kernel_width,
 input_channels,
 filters)
```

şeklindedir.


# 24. Conv2D Parametre Sayısı

Örneğimizde:

```text
3×3 kernel
1 giriş kanalı
2 filter
```

ağırlık sayısı:

```text
3 × 3 × 1 × 2 = 18
```

Bias:

```text
2
```

Toplam:

```text
20 parametre
```


In [ ]:
print(
    "Parametre:",
    conv_ornek.count_params()
)


# 25. MaxPooling2D Örneği

In [ ]:
pool_layer = layers.MaxPooling2D(
    pool_size=(
        2,
        2
    )
)

pool_tensor = np.array([
    [
        [
            [1.0],
            [3.0],
            [2.0],
            [1.0]
        ],
        [
            [4.0],
            [6.0],
            [5.0],
            [2.0]
        ],
        [
            [1.0],
            [2.0],
            [8.0],
            [7.0]
        ],
        [
            [0.0],
            [3.0],
            [4.0],
            [9.0]
        ]
    ]
])

pool_sonuc = pool_layer(
    pool_tensor
)

print(
    pool_sonuc.numpy().reshape(
        2,
        2
    )
)


# 26. Gerçek Proje: El Yazısı Rakam Sınıflandırma

İnternet bağlantısı gerektirmeyen scikit-learn Digits veri kümesini kullanacağız.

Her örnek:

```text
8 × 8
```

gri görüntüdür.

Hedef:

```text
0 - 9
```

arasındaki rakam sınıfıdır.


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()

images = digits.images.astype(
    np.float32
)

labels = digits.target

print(
    images.shape
)

print(
    labels.shape
)


# 27. İlk Görüntü

In [ ]:
plt.imshow(
    images[0],
    cmap="gray"
)

plt.title(
    f"Rakam: {labels[0]}"
)

plt.axis("off")
plt.show()


# 28. Piksel Değer Aralığı

In [ ]:
print(
    "Minimum:",
    images.min()
)

print(
    "Maksimum:",
    images.max()
)


Digits piksel değerleri 0-16 aralığındadır.


# 29. Normalizasyon

Piksel değerlerini 0-1 aralığına çevirelim.


In [ ]:
images = (
    images
    /
    16.0
)

print(
    images.min(),
    images.max()
)


# 30. Kanal Boyutunu Eklemek

Mevcut şekil:

```text
(örnek, 8, 8)
```

CNN için:

```text
(örnek, 8, 8, 1)
```

yapalım.


In [ ]:
X_cnn = np.expand_dims(
    images,
    axis=-1
)

print(
    X_cnn.shape
)


# 31. Train-Test Ayrımı

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_cnn,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print(
    "Train:",
    X_train.shape
)

print(
    "Test:",
    X_test.shape
)


# 32. CNN Mimarisi

Küçük 8×8 görüntüler için kompakt bir model oluşturacağız:

```text
8×8×1
↓
Conv2D(16, 3×3, same)
↓
MaxPooling2D(2×2)
↓
4×4×16
↓
Conv2D(32, 3×3, same)
↓
MaxPooling2D(2×2)
↓
2×2×32
↓
Flatten
↓
Dense(64)
↓
Dropout
↓
Dense(10, Softmax)
```


# 33. CNN Modelini Oluşturmak

In [ ]:
keras.utils.set_random_seed(
    42
)

cnn_model = keras.Sequential([
    keras.Input(
        shape=(
            8,
            8,
            1
        )
    ),
    layers.Conv2D(
        16,
        kernel_size=(
            3,
            3
        ),
        padding="same",
        activation="relu"
    ),
    layers.MaxPooling2D(
        pool_size=(
            2,
            2
        )
    ),
    layers.Conv2D(
        32,
        kernel_size=(
            3,
            3
        ),
        padding="same",
        activation="relu"
    ),
    layers.MaxPooling2D(
        pool_size=(
            2,
            2
        )
    ),
    layers.Flatten(),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dropout(
        0.25
    ),
    layers.Dense(
        10,
        activation="softmax"
    )
])

cnn_model.summary()


# 34. İlk Conv Katmanı Çıktısı

Girdi:

```text
8×8×1
```

`padding="same"` kullandığımız için ilk convolution sonrası:

```text
8×8×16
```

olur.


# 35. İlk Pooling Sonrası

2×2 MaxPooling:

```text
8×8×16
↓
4×4×16
```

uzamsal boyutu küçültür.


# 36. İkinci Conv ve Pooling

İkinci convolution:

```text
4×4×16
↓
4×4×32
```

İkinci pooling:

```text
4×4×32
↓
2×2×32
```

oluşturur.


# 37. Flatten

`Flatten()`:

```text
2 × 2 × 32
=
128
```

değerini tek boyutlu vektöre dönüştürür.

Bu vektör Dense katmanlara aktarılır.


# 38. Dropout

Dropout eğitim sırasında belirli nöron çıktılarını rastgele devre dışı bırakarak overfitting'i azaltmaya yardımcı olabilir.

Bu projede:

```python
Dropout(0.25)
```

kullanıyoruz.

Dropout tahmin sırasında eğitimdeki gibi rastgele nöron kapatmaz.


# 39. Softmax Çıkış

10 rakam sınıfımız olduğu için:

```python
Dense(
    10,
    activation="softmax"
)
```

kullanıyoruz.


# 40. Modeli Compile Etmek

In [ ]:
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)


Etiketlerimiz:

```text
0, 1, 2, ..., 9
```

şeklinde integer olduğu için `sparse_categorical_crossentropy` uygundur.


# 41. EarlyStopping

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


# 42. CNN Modelini Eğitmek

In [ ]:
history = cnn_model.fit(
    X_train,
    y_train,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    callbacks=[
        early_stop
    ],
    verbose=0
)

print(
    "Gerçekleşen epoch:",
    len(
        history.history[
            "loss"
        ]
    )
)


# 43. Eğitim Accuracy Grafiği

In [ ]:
plt.plot(
    history.history[
        "accuracy"
    ],
    label="Train"
)

plt.plot(
    history.history[
        "val_accuracy"
    ],
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()
plt.show()


# 44. Eğitim Loss Grafiği

In [ ]:
plt.plot(
    history.history[
        "loss"
    ],
    label="Train"
)

plt.plot(
    history.history[
        "val_loss"
    ],
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Loss")
plt.legend()
plt.show()


# 45. Final Test Değerlendirmesi

In [ ]:
test_loss, test_accuracy = (
    cnn_model.evaluate(
        X_test,
        y_test,
        verbose=0
    )
)

print(
    "Test Loss:",
    test_loss
)

print(
    "Test Accuracy:",
    test_accuracy
)


# 46. Test Tahminleri

In [ ]:
test_olasilik = (
    cnn_model.predict(
        X_test,
        verbose=0
    )
)

print(
    test_olasilik.shape
)


Her test örneği için 10 sınıf skoru üretilmiştir.


# 47. Sınıf Tahminlerini Bulmak

In [ ]:
y_pred = np.argmax(
    test_olasilik,
    axis=1
)

print(
    y_pred[:20]
)


# 48. Accuracy'yi Scikit-learn ile Kontrol Etmek

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print(
    accuracy_score(
        y_test,
        y_pred
    )
)


# 49. Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        digits=3
    )
)


# 50. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=range(10)
).plot()

plt.title(
    "CNN Confusion Matrix"
)

plt.show()


# 51. Yanlış Tahmin Sayısı

In [ ]:
yanlis_indexler = np.where(
    y_pred
    !=
    y_test
)[0]

print(
    "Yanlış tahmin:",
    len(
        yanlis_indexler
    )
)


# 52. İlk Yanlış Tahmini Görmek

In [ ]:
if len(
    yanlis_indexler
) > 0:
    i = yanlis_indexler[0]

    plt.imshow(
        X_test[
            i,
            :,
            :,
            0
        ],
        cmap="gray"
    )

    plt.title(
        f"Gerçek: {y_test[i]} - Tahmin: {y_pred[i]}"
    )

    plt.axis("off")
    plt.show()
else:
    print(
        "Yanlış tahmin bulunamadı."
    )


# 53. Birkaç Yanlış Tahmini İncelemek

In [ ]:
for i in yanlis_indexler[:5]:
    plt.figure()

    plt.imshow(
        X_test[
            i,
            :,
            :,
            0
        ],
        cmap="gray"
    )

    plt.title(
        f"Gerçek: {y_test[i]} - Tahmin: {y_pred[i]}"
    )

    plt.axis("off")
    plt.show()


Hata analizi CNN modelinde de önemlidir.

Sadece toplam accuracy'ye bakmak yerine hangi rakamların karıştırıldığını incelemeliyiz.


# 54. Tek Görüntü İçin Tahmin

In [ ]:
ornek = X_test[:1]

olasilik = (
    cnn_model.predict(
        ornek,
        verbose=0
    )[0]
)

tahmin = int(
    np.argmax(
        olasilik
    )
)

print(
    "Gerçek:",
    y_test[0]
)

print(
    "Tahmin:",
    tahmin
)


# 55. İlk Üç Sınıf Adayı

In [ ]:
sirali_index = np.argsort(
    olasilik
)[::-1]

for index in sirali_index[:3]:
    print(
        "Rakam:",
        int(index),
        "Skor:",
        round(
            float(
                olasilik[
                    index
                ]
            ),
            4
        )
    )


Softmax çıktıları modelin ürettiği sınıf skorlarıdır.

Bunları gerçek dünyada otomatik olarak kusursuz kalibre edilmiş güven yüzdesi gibi yorumlamamalıyız.


# 56. İlk Conv Katmanının Feature Map'lerini Görmek

Eğitilmiş modelin ilk convolution katmanındaki çıktıları inceleyebiliriz.


In [ ]:
ilk_conv = (
    cnn_model.layers[0]
)

print(
    ilk_conv.name
)

print(
    ilk_conv.output.shape
)


Sequential modelimizde `keras.Input` ayrı bir layer olarak `model.layers` listesinde yer almayabilir. Bu nedenle ilk kayıtlı layer Conv2D olabilir.


# 57. Feature Extractor Modeli

İlk Conv2D katmanının çıktısını veren ara model oluşturalım.


In [ ]:
feature_model = keras.Model(
    inputs=cnn_model.inputs,
    outputs=ilk_conv.output
)

feature_maps = feature_model.predict(
    X_test[:1],
    verbose=0
)

print(
    feature_maps.shape
)


Şekil yaklaşık olarak:

```text
1 × 8 × 8 × 16
```

olur.

Yani 16 farklı feature map vardır.


# 58. İlk Feature Map

In [ ]:
plt.imshow(
    feature_maps[
        0,
        :,
        :,
        0
    ],
    cmap="gray"
)

plt.title("İlk Conv Filter Feature Map")
plt.axis("off")
plt.show()


# 59. İlk Dört Feature Map'i Ayrı Ayrı Görmek

In [ ]:
for kanal in range(4):
    plt.figure()

    plt.imshow(
        feature_maps[
            0,
            :,
            :,
            kanal
        ],
        cmap="gray"
    )

    plt.title(
        f"Feature Map {kanal}"
    )

    plt.axis("off")
    plt.show()


Farklı filtreler aynı görüntünün farklı örüntülerine tepki verebilir.

Filtreleri doğrudan insan kavramlarıyla kesin biçimde isimlendirmek her zaman mümkün değildir.


# 60. Conv Kernel Ağırlıklarını İncelemek

In [ ]:
conv_weights, conv_bias = (
    ilk_conv.get_weights()
)

print(
    "Kernel:",
    conv_weights.shape
)

print(
    "Bias:",
    conv_bias.shape
)


# 61. İlk Öğrenilmiş Kernel'i Görmek

İlk giriş kanalı ve ilk filter:


In [ ]:
ilk_kernel = conv_weights[
    :,
    :,
    0,
    0
]

print(
    ilk_kernel
)

plt.imshow(
    ilk_kernel,
    cmap="gray"
)

plt.title("Öğrenilmiş 3x3 Kernel")
plt.axis("off")
plt.show()


CNN'deki filtre değerlerini eğitim sırasında model öğrenmiştir.


# 62. Dense Ağ ile CNN Karşılaştırması

Aynı Digits verisinde önceki dersimizde Dense ağ kullanmıştık.

Karşılaştırma için küçük bir Dense model oluşturalım.


In [ ]:
X_dense = (
    X_cnn.reshape(
        len(
            X_cnn
        ),
        -1
    )
)

X_train_dense, X_test_dense, y_train_dense, y_test_dense = train_test_split(
    X_dense,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)


# 63. Dense Model

In [ ]:
keras.utils.set_random_seed(
    42
)

dense_model = keras.Sequential([
    keras.Input(
        shape=(64,)
    ),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dense(
        32,
        activation="relu"
    ),
    layers.Dense(
        10,
        activation="softmax"
    )
])

dense_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)

dense_model.fit(
    X_train_dense,
    y_train_dense,
    validation_split=0.20,
    epochs=25,
    batch_size=32,
    verbose=0
)

dense_loss, dense_accuracy = (
    dense_model.evaluate(
        X_test_dense,
        y_test_dense,
        verbose=0
    )
)

print(
    "Dense Accuracy:",
    dense_accuracy
)

print(
    "CNN Accuracy:",
    test_accuracy
)


Bu küçük veri kümesinde iki model de güçlü sonuçlar verebilir.

CNN'in asıl avantajı daha büyük ve uzamsal olarak zengin görüntü problemlerinde daha belirgin hale gelir.


# 64. CNN Parametre Sayısı

In [ ]:
print(
    "CNN parametre:",
    cnn_model.count_params()
)

print(
    "Dense parametre:",
    dense_model.count_params()
)


Model seçerken yalnızca accuracy değil:

- parametre sayısı,
- veri miktarı,
- tahmin süresi,
- eğitim süresi,
- donanım,
- genelleme başarısı

gibi faktörler de önemlidir.


# 65. Data Augmentation Nedir?

Görüntü eğitiminde mevcut görüntülere küçük dönüşümler uygulayarak yeni varyasyonlar oluşturabiliriz.

Örnekler:

- küçük döndürme,
- yatay/dikey kaydırma,
- zoom,
- bazı problemlerde flip.

Bu işleme **data augmentation** denir.


# 66. Her Augmentation Her Probleme Uygun Değildir

Örneğin el yazısı rakamlarında:

```text
6
```

görüntüsünü dikey veya yatay çevirmek anlamsız veya başka bir sembole benzeyen örnek oluşturabilir.

Bu nedenle augmentation seçimi problem bilgisini dikkate almalıdır.


# 67. Küçük Rakam Augmentation Katmanı

Digits için yalnızca küçük döndürme ve küçük translation kullanalım.


In [ ]:
augmentation = keras.Sequential([
    layers.RandomRotation(
        factor=0.05,
        fill_mode="constant"
    ),
    layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05,
        fill_mode="constant"
    )
])

print(
    augmentation
)


# 68. Augmented Örnek Görüntü

In [ ]:
augmented = augmentation(
    X_train[:1],
    training=True
).numpy()

plt.imshow(
    augmented[
        0,
        :,
        :,
        0
    ],
    cmap="gray"
)

plt.title("Augmented Rakam")
plt.axis("off")
plt.show()


# 69. Augmentation'lı CNN Mimarisi

Data augmentation katmanını modelin başlangıcına ekleyebiliriz.


In [ ]:
keras.utils.set_random_seed(
    42
)

aug_cnn = keras.Sequential([
    keras.Input(
        shape=(
            8,
            8,
            1
        )
    ),
    augmentation,
    layers.Conv2D(
        16,
        3,
        padding="same",
        activation="relu"
    ),
    layers.MaxPooling2D(
        2
    ),
    layers.Conv2D(
        32,
        3,
        padding="same",
        activation="relu"
    ),
    layers.MaxPooling2D(
        2
    ),
    layers.Flatten(),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dropout(
        0.25
    ),
    layers.Dense(
        10,
        activation="softmax"
    )
])

aug_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

aug_cnn.summary()


Bu model eğitim sırasında augmentation uygulayabilir.

Validation ve tahmin sırasında random augmentation katmanları normal eğitim davranışındaki gibi rastgele dönüşüm üretmez.


# 70. Augmentation Ne Zaman Faydalı Olabilir?

Özellikle:

- veri azsa,
- gerçek kullanımda küçük konum/açı değişimleri bekleniyorsa,
- dönüşüm sınıf anlamını değiştirmiyorsa

yararlı olabilir.

Ancak gereksiz veya aşırı augmentation modeli zorlaştırabilir.


# 71. GlobalAveragePooling2D Kavramı

Bazı CNN mimarilerinde `Flatten()` yerine:

```python
GlobalAveragePooling2D()
```

kullanılabilir.

Bu katman her feature map'in uzamsal ortalamasını alarak kanal başına tek değer üretir.


# 72. GlobalAveragePooling2D Örneği

In [ ]:
gap_model = keras.Sequential([
    keras.Input(
        shape=(
            8,
            8,
            1
        )
    ),
    layers.Conv2D(
        32,
        3,
        padding="same",
        activation="relu"
    ),
    layers.MaxPooling2D(
        2
    ),
    layers.Conv2D(
        64,
        3,
        padding="same",
        activation="relu"
    ),
    layers.GlobalAveragePooling2D(),
    layers.Dense(
        10,
        activation="softmax"
    )
])

gap_model.summary()


GlobalAveragePooling2D bazı mimarilerde Dense bölüme giden parametre sayısını azaltabilir.


# 73. BatchNormalization Kavramına Giriş

`BatchNormalization` ara aktivasyonların dağılımını normalize etmeye yardımcı olan öğrenilebilir bir katmandır.

Bazı derin CNN mimarilerinde eğitim kararlılığını artırabilir.

Her küçük modele zorunlu olarak eklenmesi gerekmez.


In [ ]:
bn_ornek = keras.Sequential([
    keras.Input(
        shape=(
            8,
            8,
            1
        )
    ),
    layers.Conv2D(
        16,
        3,
        padding="same"
    ),
    layers.BatchNormalization(),
    layers.Activation(
        "relu"
    ),
    layers.MaxPooling2D(
        2
    )
])

bn_ornek.summary()


# 74. CNN Modelini Kaydetmek

Eğitilmiş CNN'i `.keras` dosyasına kaydedelim.


In [ ]:
MODEL_DOSYASI = (
    "31-cnn-rakam-modeli.keras"
)

cnn_model.save(
    MODEL_DOSYASI
)

print(
    "Model kaydedildi:",
    MODEL_DOSYASI
)


# 75. CNN Modelini Yeniden Yüklemek

In [ ]:
cnn_yuklu = (
    keras.models.load_model(
        MODEL_DOSYASI
    )
)

cnn_yuklu.summary()


# 76. Yüklenen Modelle Tahmin

In [ ]:
kontrol_olasilik = (
    cnn_yuklu.predict(
        X_test[:1],
        verbose=0
    )
)

print(
    "Tahmin:",
    np.argmax(
        kontrol_olasilik,
        axis=1
    )[0]
)

print(
    "Gerçek:",
    y_test[0]
)


# 77. Görüntü Tahmin Fonksiyonu

8×8 gri görüntüyü alıp CNN tahmini üreten fonksiyon yazalım.


In [ ]:
def cnn_rakam_tahmin(
    model,
    goruntu
):
    goruntu = np.array(
        goruntu,
        dtype=np.float32
    )

    if goruntu.shape != (
        8,
        8
    ):
        raise ValueError(
            "Görüntü 8x8 olmalıdır."
        )

    if goruntu.max() > 1:
        goruntu = (
            goruntu
            /
            16.0
        )

    girdi = goruntu.reshape(
        1,
        8,
        8,
        1
    )

    olasilik = model.predict(
        girdi,
        verbose=0
    )[0]

    tahmin = int(
        np.argmax(
            olasilik
        )
    )

    return {
        "tahmin": tahmin,
        "skorlar": olasilik
    }


In [ ]:
sonuc = cnn_rakam_tahmin(
    cnn_yuklu,
    digits.images[10]
)

print(
    "Gerçek:",
    digits.target[10]
)

print(
    "Tahmin:",
    sonuc[
        "tahmin"
    ]
)


# 78. OpenCV ile CNN'i Birleştirmek

Bir önceki görüntü işleme dersimizde dış görüntüyü:

```text
oku
↓
gri ton
↓
resize
↓
normalize
```

edebiliyorduk.

CNN ile gerçek uygulamada:

```text
Fotoğraf
↓
OpenCV
↓
Modelin beklediği görüntü biçimi
↓
CNN
↓
Tahmin
```

zinciri kurulabilir.


# 79. Dış Görüntü Neden Doğrudan Digits CNN'e Verilemez?

Model şu veriyle eğitildi:

```text
8×8
gri
0-16 kaynak piksel ölçeği
belirli rakam yerleşimi
```

Normal bir telefon fotoğrafı:

```text
1920×1080
RGB
0-255
arka planlı
farklı ölçekte
```

olabilir.

Bu nedenle:

- rakamı bulma,
- kırpma,
- merkezleme,
- gri tona çevirme,
- 8×8'e küçültme,
- uygun piksel ölçeği

gibi işlemler gerekir.


# 80. CNN Uygulaması İçin Ön İşleme Fonksiyonu Taslağı

Gerçek dış görüntü için örnek akış:


In [ ]:
def cnn_on_isleme_taslagi(
    gri_goruntu
):
    goruntu = np.array(
        gri_goruntu,
        dtype=np.float32
    )

    goruntu = np.clip(
        goruntu,
        0,
        255
    )

    goruntu = (
        goruntu
        /
        255.0
    )

    goruntu = tf.image.resize(
        goruntu[
            ...,
            np.newaxis
        ],
        (
            8,
            8
        )
    ).numpy()

    return goruntu.reshape(
        1,
        8,
        8,
        1
    )


Bu yalnızca genel ön işleme örneğidir.

Digits veri kümesinin gerçek görüntü dağılımını tam olarak taklit etmez. Gerçek kamera rakamları için ek merkezleme, kontrast ve piksel dağılımı uyarlaması gerekir.


# 81. CNN'de Feature Extraction

CNN'in convolution katmanları ham görüntüden özellik çıkarma işini büyük ölçüde kendi öğrenebilir.

İlk katmanlar daha basit:

- kenar,
- yön,
- yerel doku

benzeri örüntülere duyarlı olabilir.

Daha derin katmanlar bu düşük seviyeli özellikleri birleştirerek daha karmaşık temsiller öğrenebilir.


# 82. Klasik Görüntü İşleme ile CNN Farkı

### Klasik yaklaşım

Biz belirleriz:

```text
Canny
Contour
Alan
Çevre
En-Boy Oranı
```

Ardından model bu özellikleri kullanır.

### CNN yaklaşımı

Model convolution filtrelerini eğitim sırasında öğrenir.

```text
Piksel
↓
Conv Katmanları
↓
Öğrenilmiş Özellikler
↓
Sınıflandırma
```


# 83. CNN Her Zaman Klasik Yöntemden İyi midir?

Hayır.

Küçük ve basit bir problemde:

- contour,
- geometrik özellik,
- Random Forest

çok daha kolay ve yeterli olabilir.

CNN daha fazla:

- veri,
- hesaplama,
- eğitim,
- model yönetimi

gerektirebilir.

Yöntem problem gereksinimine göre seçilmelidir.


# 84. CNN ve Veri Miktarı

Derin öğrenme modelleri özellikle büyük veri kümelerinden faydalanabilir.

Veri az olduğunda:

- overfitting,
- veri çeşitliliği eksikliği,
- zayıf genelleme

sorunları oluşabilir.

İlerleyen derslerde **transfer learning** ile az veride önceden eğitilmiş modellerden yararlanmayı öğreneceğiz.


# 85. Transfer Learning'e İlk Bakış

Büyük bir görüntü veri kümesinde önceden eğitilmiş modelin öğrendiği özellikleri kendi problemimizde kullanabiliriz.

Örnek akış:

```text
Önceden Eğitilmiş CNN
↓
Feature Extractor
↓
Yeni Sınıflandırma Katmanı
↓
Kendi Veri Kümen
```

Bu yaklaşıma transfer learning denir.


# 86. Fine-Tuning Kavramı

Transfer learning sonrasında modelin bazı önceden eğitilmiş katmanlarını küçük learning rate ile yeniden eğitmeye **fine-tuning** denebilir.

Bu konu daha büyük görüntü modelleriyle çalışırken ayrıntılı işlenecektir.


# 87. CNN Modelinde Overfitting

Örnek işaret:

```text
Train Accuracy      → 0.99
Validation Accuracy → 0.85
Validation Loss     → yükseliyor
```

Olası önlemler:

- EarlyStopping,
- Dropout,
- data augmentation,
- daha küçük model,
- daha fazla veri,
- regularization.


# 88. CNN Modelinde Underfitting

Eğer hem train hem validation başarısı düşükse:

- model kapasitesi yetersiz,
- epoch az,
- learning rate uygun değil,
- veri ön işleme hatalı,
- veri yeterince bilgi içermiyor

olabilir.

Sadece katman eklemek her zaman çözüm değildir.


# 89. Kernel Boyutu

Yaygın kernel boyutlarından biri:

```text
3×3
```

dür.

Daha büyük kernel:

- daha geniş bölgeyi aynı anda görür,
- daha fazla parametre oluşturabilir.

Birçok modern CNN küçük kernel'leri ardışık kullanarak daha derin özellik çıkarımı yapar.


# 90. Filter Sayısı

Örneğin:

```text
Conv2D(16)
Conv2D(32)
Conv2D(64)
```

derinleştikçe daha fazla feature map kullanılabilir.

Ancak filter sayısının artması:

- parametre,
- bellek,
- hesaplama

maliyetini de artırabilir.


# 91. Feature Map Boyutlarını Takip Etmek

CNN tasarlarken her katmandaki şekli takip etmek önemlidir.

Örnek:

```text
32×32×3
↓ Conv same, 32 filters
32×32×32
↓ Pool 2×2
16×16×32
↓ Conv same, 64 filters
16×16×64
↓ Pool
8×8×64
```

Bu takip model mimarisini anlamayı kolaylaştırır.


# 92. Model Özeti Neden Önemlidir?

`model.summary()` bize:

- katman sırası,
- output shape,
- parametre sayısı

bilgilerini verir.

CNN kurduktan sonra mutlaka model özetini kontrol etmek iyi bir alışkanlıktır.


# 93. Veri Augmentation'da Etiket Korunmalı

Bir dönüşüm uyguladığımızda sınıf aynı kalmalıdır.

Örneğin bir kedi fotoğrafını küçük açıyla döndürmek hâlâ kedi olabilir.

Ancak bazı problemlerde:

- dikey çevirme,
- büyük dönüş,
- renk değiştirme

etiketi bozabilir.

Augmentation gerçek dünyadaki olası değişimleri temsil etmelidir.


# 94. Görüntü Sınıflandırmada Veri Gizliliği

Gerçek projelerde görüntüler:

- yüz,
- plaka,
- belge,
- ev içi görüntü,
- konum,
- kişisel eşya

gibi hassas bilgi içerebilir.

Verinin:

- kullanım izni,
- saklanması,
- anonimleştirilmesi,
- erişim yetkileri

dikkate alınmalıdır.


# 95. Model Hatasının Etkisi

Rakam tanıma gibi düşük riskli eğitim projesindeki yanlış tahmin ile:

- sağlık görüntüsü,
- trafik sistemi,
- güvenlik sistemi

gibi yüksek etkili alanlardaki yanlış tahmin aynı değildir.

Yapay zeka modelini değerlendirirken yalnızca teknik başarı değil, hata sonucunun etkisi de incelenmelidir.


# 96. CNN Proje Akışı

```text
Görüntü Veri Kümesi
↓
Boyut / Kanal Kontrolü
↓
Normalizasyon
↓
Train / Validation / Test
↓
Conv2D
↓
Activation
↓
Pooling
↓
Conv2D
↓
Pooling
↓
Flatten / GlobalAveragePooling
↓
Dense
↓
Softmax
↓
Training
↓
Learning Curve
↓
Test
↓
Confusion Matrix
↓
Hata Analizi
↓
Model Kaydı
```


# 97. Ders Özeti

Bu derste:

- CNN,
- convolution,
- kernel,
- filter,
- stride,
- padding,
- feature map,
- Conv2D,
- MaxPooling2D,
- Flatten,
- GlobalAveragePooling2D,
- Dropout,
- BatchNormalization kavramı,
- görüntü tensörü,
- channel boyutu,
- normalizasyon,
- CNN mimarisi,
- multi-class classification,
- softmax,
- sparse categorical crossentropy,
- EarlyStopping,
- learning curve,
- confusion matrix,
- hata analizi,
- feature map görselleştirme,
- öğrenilmiş kernel,
- Dense-CNN karşılaştırması,
- data augmentation,
- transfer learning kavramı,
- fine-tuning kavramı,
- model kaydetme,
- görüntü ön işleme

konularını öğrendik.


# 98. Mini Uygulamalar

1. 5×5 görüntü ve 3×3 kernel oluşturun.
2. Tek convolution işlemini elle hesaplayın.
3. Basit convolution fonksiyonu yazın.
4. Feature map'i görselleştirin.
5. 4×4 matrise manuel Max Pooling uygulayın.
6. Keras `Conv2D` ile iki filter oluşturun.
7. Conv2D çıktı boyutunu inceleyin.
8. Conv2D parametre sayısını hesaplayın.
9. `MaxPooling2D` örneği oluşturun.
10. Digits verisini 8×8×1 şekline dönüştürün.
11. Piksel verisini 0-1 aralığına normalize edin.
12. Tek Conv2D katmanlı CNN oluşturun.
13. İki Conv2D katmanlı CNN oluşturun.
14. Model `summary()` çıktısını analiz edin.
15. CNN modelini eğitin.
16. Train-validation accuracy grafiği çizin.
17. Train-validation loss grafiği çizin.
18. Test accuracy hesaplayın.
19. Classification report oluşturun.
20. Confusion matrix oluşturun.
21. Yanlış tahmin edilen görüntüleri gösterin.
22. İlk Conv katmanının feature map'lerini gösterin.
23. Data augmentation katmanı oluşturun.
24. CNN ile Dense modelin parametre ve başarısını karşılaştırın.
25. CNN modelini `.keras` dosyasına kaydedip yeniden yükleyin.


# 99. Yapay Zeka Proje Görevi

Bir **CNN Görüntü Sınıflandırma Sistemi** geliştirin.

Projede en az:

- görüntü veri kümesi,
- en az 3 sınıf,
- görüntü boyutu kontrolü,
- channel boyutu,
- normalizasyon,
- train-test ayrımı,
- en az 2 Conv2D katmanı,
- MaxPooling2D,
- ReLU,
- Dropout,
- Softmax,
- uygun loss,
- validation,
- EarlyStopping,
- learning curve,
- accuracy,
- classification report,
- confusion matrix,
- yanlış tahmin analizi,
- yeni görüntü tahmini,
- `.keras` model kaydı

bulunsun.

Ek geliştirme:

- data augmentation,
- feature map görselleştirme,
- OpenCV ön işleme,
- Tkinter arayüzü,
- Flask web uygulaması

özelliklerinden biri eklenebilir.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin şu CNN zincirini anlayıp uygulayabilmesi hedeflenmektedir:

**Görüntü**

↓

**Conv2D**

↓

**Feature Map**

↓

**ReLU**

↓

**MaxPooling**

↓

**Daha Derin Conv Katmanları**

↓

**Flatten / GlobalAveragePooling**

↓

**Dense**

↓

**Softmax**

↓

**Tahmin**

Artık öğrenciler yalnızca görüntüleri işlemekle kalmıyor; convolution filtrelerini öğrenen gerçek bir CNN modeli oluşturup görüntü sınıflandırma uygulaması geliştirebiliyor.

Bir sonraki derste daha büyük ve gerçek görüntü problemleri için **Transfer Learning ve Önceden Eğitilmiş Modeller** konusuna geçeceğiz.
